# 34. 선례 라이브러리(Precedent Library) 프로토타입

## 이번 노트북에서 할 것
- 긍정 선례(승인약물 매칭쌍) + 부정 선례(철수약물) + 정량 선례(GtoPdb)를
  하나의 구조화된 라이브러리로 정리
- 선례 검색 함수 구현 (규칙 이름으로 관련 선례 찾기)
- 약리학/독성학 에이전트 프롬프트에 선례를 주입하도록 확장
- 선례 유무에 따라 판단이 실제로 달라지는지 확인 (BCP 케이스로 테스트)

## 배경 (33까지)
- 라이브러리 34개 규칙, 커버리지 33.2%
- 3-에이전트(독성학/의약화학/약리학) + 조정자 완성, 30개 배치 검증
  (0% 자동승인, 44.4% 치환재검토, 55.6% 사람검토필요)
- 핵심 발견: 정량지표(형태보존 등)가 긍정적이어도 LLM이 수소결합/전자
  분포 등 지표 미반영 요소로 추가 우려 제기 - 두 계층의 상호보완성 확인
- README, case study, ablation, SA score, 로드맵 문서 전부 커밋 완료
- 오늘(이 노트북)은 fine-tuning 없이 "선례를 프롬프트에 주입"하는
  few-shot 방식으로 판단 근거를 강화하는 프로토타입 제작
- test set은 여전히 미사용

## 다음 계획
- 제안서 최종 정리 (학생, 여유시간에 진행)
- Test set 최종 검증 (모든 확장 마무리 후, 학생 승인 시)

In [1]:
# 셀 1
!pip install rdkit -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q
!pip install openai -q
!pip install chembl_webresource_client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 764.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00


In [2]:
# 셀 2
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 393, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 393 (delta 67), reused 103 (delta 42), pack-reused 260 (from 1)
Receiving objects: 100% (393/393), 4.64 MiB | 10.97 MiB/s, done.
Resolving deltas: 100% (204/204), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
# 셀 3
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [4]:
# 셀 4
import importlib, json
from rdkit import Chem

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.agent

from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix
from src.tools.agent import _call_llm, _parse_json_response

print(f"라이브러리 규칙 수: {len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])}")

라이브러리 규칙 수: 35


In [5]:
# 셀 5 — Qwen 연결
from openai import OpenAI
dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")
print("준비 완료")

준비 완료


In [6]:
PRECEDENT_LIBRARY = [
    # 긍정 선례 (승인약물 매칭쌍) - 오늘 ChEMBL로 확인한 것
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},

    # 정량 선례 (GtoPdb 실측 데이터)
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},

    # 부정 선례 (철수/제한 약물) - 잘 알려진 사례, 향후 문헌 확인 필요
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
]

print(f"선례 {len(PRECEDENT_LIBRARY)}건 로드")
for p in PRECEDENT_LIBRARY:
    print(f"  [{p['rule']}] {p['type']}")

선례 6건 로드
  [Thiocarbonyl_group] 긍정_승인약물쌍
  [catechol] 정량_활성데이터
  [hydroxamic_acid] 부정_참고사례_검증필요
  [beta-keto/anhydride] 긍정_통계검증결과
  [Michael_acceptor_1] 위험=메커니즘_참고
  [alkyl_halide] 위험=메커니즘_참고


In [7]:
def get_precedents(rule_name):
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])

# 확인
print(get_precedents("catechol"))
print()
print(get_precedents("nitro_group"))  # 선례 없는 규칙

- [정량_활성데이터] 도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침.

None


In [8]:
def ask_pharmacology_agent_v2(client, model_name, original_smiles, fixed_smiles, rule_name, metrics, classification, client_type="openai_compatible"):
    """선례 라이브러리를 참고자료로 추가한 약리학 에이전트."""
    precedents = get_precedents(rule_name)
    precedent_text = f"\n실제 참고 선례:\n{precedents}\n" if precedents else "\n(관련 선례 없음)\n"

    prompt = f"""당신은 약리학 전문가입니다. 다음 분자 치환에 대한 정량 분석
결과를 검토하고, 표적 단백질과의 상호작용(활성) 관점에서 최종 코멘트를
작성해주세요.

원본: {original_smiles}
치환 후: {fixed_smiles}

정량 분석 결과:
{chr(10).join(classification['details'])}

규칙기반 1차 판정: {classification['verdict']}
{precedent_text}
위 실제 선례가 있다면 이를 판단에 명시적으로 반영하세요(예: 실제 승인약물
사례가 있다면 그 방향을 더 신뢰할 수 있고, 부정적 참고사례가 있다면 더
신중해야 합니다). 아래 JSON으로만 답하세요.

{{"agree_with_verdict": true/false, "final_comment": "1-2문장 코멘트 (선례를 언급했다면 명시)",
"human_review_needed": true/false, "precedent_used": true/false}}
"""
    text = _call_llm(client, model_name, prompt, client_type)
    fallback = {"agree_with_verdict": True, "final_comment": "자동판정 근거 참고",
                "human_review_needed": classification.get('shape_ok') is False, "precedent_used": False}
    return _parse_json_response(text, fallback)

print("선례 통합 약리학 에이전트 준비 완료")

선례 통합 약리학 에이전트 준비 완료


In [10]:
from rdkit.Chem import rdFingerprintGenerator, Descriptors, Descriptors3D, DataStructs, QED, AllChem

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

import urllib.request, sys
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py",
    "sascorer.py")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz",
    "fpscores.pkl.gz")
sys.path.append('.')
import sascorer


def compute_activity_preservation_metrics(original_smiles, fixed_smiles):
    mol_o = Chem.MolFromSmiles(original_smiles)
    mol_f = Chem.MolFromSmiles(fixed_smiles)
    if mol_o is None or mol_f is None:
        return None

    fp_o = _generator.GetFingerprint(mol_o)
    fp_f = _generator.GetFingerprint(mol_f)
    tanimoto = DataStructs.TanimotoSimilarity(fp_o, fp_f)

    qed_o, qed_f = QED.qed(mol_o), QED.qed(mol_f)
    logp_o, logp_f = Descriptors.MolLogP(mol_o), Descriptors.MolLogP(mol_f)
    sa_o, sa_f = sascorer.calculateScore(mol_o), sascorer.calculateScore(mol_f)

    def get_3d(mol):
        m = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(m, randomSeed=42) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(m)
        return m

    m3d_o, m3d_f = get_3d(mol_o), get_3d(mol_f)
    shape_available = m3d_o is not None and m3d_f is not None

    result = {
        "tanimoto": tanimoto,
        "delta_qed": qed_f - qed_o,
        "delta_logp": logp_f - logp_o,
        "delta_sa_score": sa_f - sa_o,
        "shape_available": shape_available,
    }

    if shape_available:
        rog_o = Descriptors3D.RadiusOfGyration(m3d_o)
        rog_f = Descriptors3D.RadiusOfGyration(m3d_f)
        result["delta_rog_pct"] = (rog_f - rog_o) / rog_o * 100 if rog_o != 0 else None

    return result


def classify_activity_risk_v3(metrics):
    if metrics is None:
        return {"verdict": "판정 불가", "details": []}

    details = []
    warnings = []

    conn_ok = metrics["tanimoto"] >= 0.5
    details.append(f"2D 연결성: {'유사' if conn_ok else '상이'} (Tanimoto {metrics['tanimoto']:.3f})")

    shape_ok = None
    if metrics["shape_available"] and metrics.get("delta_rog_pct") is not None:
        shape_ok = abs(metrics["delta_rog_pct"]) < 15
        details.append(f"3D 형태: {'보존' if shape_ok else '변화'} (회전반경 {metrics['delta_rog_pct']:+.1f}%)")
    else:
        details.append("3D 형태: 계산 불가")

    qed_ok = abs(metrics["delta_qed"]) < 0.1
    details.append(f"약물유사성(QED): {'유지' if qed_ok else '변화'} ({metrics['delta_qed']:+.3f})")
    if not qed_ok:
        warnings.append("QED 변화")

    logp_ok = abs(metrics["delta_logp"]) < 1.0
    details.append(f"소수성(LogP): {'유지' if logp_ok else '변화'} ({metrics['delta_logp']:+.3f})")
    if not logp_ok:
        warnings.append("LogP 변화")

    sa_ok = metrics["delta_sa_score"] < 0.5
    details.append(f"합성용이성(SA): {'유지/개선' if sa_ok else '악화'} ({metrics['delta_sa_score']:+.3f})")
    if not sa_ok:
        warnings.append("합성난이도 증가")

    if shape_ok is None:
        verdict = "3D 형태 계산 불가 — 2D 지표만으로 판단, 신뢰도 낮음"
    elif shape_ok:
        if conn_ok:
            verdict = "구조·형태 모두 보존 — 활성 유지 가능성 높음"
        else:
            verdict = "2D 연결성은 크게 바뀌었으나 3D 형태는 보존됨 (bioisostere 가능성) — 활성 유지 기대"
    else:
        verdict = "3D 형태 자체가 크게 변화 — 표적 결합 형태 훼손 우려, 사람 검토 필요"

    if warnings:
        verdict += f" [보조 경고: {', '.join(warnings)}]"

    return {"verdict": verdict, "details": details, "shape_ok": shape_ok, "warnings": warnings}


print("지표 함수 준비 완료")

지표 함수 준비 완료


In [11]:
# 카테콜아민 유사 분자로 catechol 치환 테스트 (긍정 정량선례 있음)
test_smiles_catechol = "NCCc1ccc(O)c(O)c1"
fixed_catechol = propose_fix(test_smiles_catechol, "catechol", candidate_idx=0)

metrics_cat = compute_activity_preservation_metrics(test_smiles_catechol, fixed_catechol['new_smiles'])
classification_cat = classify_activity_risk_v3(metrics_cat)

result_with_precedent = ask_pharmacology_agent_v2(
    client_qwen, "qwen3.8-max-preview", test_smiles_catechol, fixed_catechol['new_smiles'],
    "catechol", metrics_cat, classification_cat, "openai_compatible"
)
print("=== 선례 있음(catechol) ===")
print(result_with_precedent)

=== 선례 있음(catechol) ===
{'agree_with_verdict': False, 'final_comment': '참조 선례에서 도파민의 카테콜 골격이 D1/D2/D3 수용체의 단자릿수 nM 결합에 필수적이므로, 한쪽 OH를 메톡시로 치환한 것은 정량적 구조 유사성에도 불구하고 활성 감소 가능성이 큽니다. 따라서 규칙기반의 ‘활성 유지 가능성 높음’ 판정보다는 실험적 검증이 필요합니다.', 'human_review_needed': True, 'precedent_used': True}


In [12]:
import requests

base_url = "https://www.guidetopharmacology.org/services"

def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()

def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

print("GtoPdb 함수 준비 완료")

GtoPdb 함수 준비 완료


In [13]:
from chembl_webresource_client.new_client import new_client
molecule = new_client.molecule

sulfasalazine_result = list(molecule.filter(pref_name__iexact="SULFASALAZINE").only(
    ['molecule_structures', 'pref_name', 'max_phase']))
print("ChEMBL 결과:", sulfasalazine_result)

sulfasalazine_gtopdb = search_ligand("sulfasalazine")
print("\nGtoPdb 결과:", sulfasalazine_gtopdb[:3] if sulfasalazine_gtopdb else "없음")

ChEMBL 결과: [{'max_phase': '4.0', 'molecule_structures': {'canonical_smiles': 'O=C(O)c1cc(/N=N/c2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O', 'molfile': '\n     RDKit          2D\n\n 28 30  0  0  0  0  0  0  0  0999 V2000\n    6.4542   -4.4375    0.0000 S   0  0  0  0  0  0  0  0  0  0  0  0\n    7.0667   -4.0833    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    1.5542   -3.0333    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    0.9417   -3.3875    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    5.8375   -4.0833    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    3.3875   -3.3792    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    4.0000   -3.0250    0.0000 N   0  0  0  0  0  0  0  0  0  0  0  0\n    7.6792   -4.4375    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    6.8042   -5.0500    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    6.0917   -5.0500    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n    1.5500   -2.3208    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n    2.1

In [14]:
PRECEDENT_LIBRARY.append({
    "rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
    "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                    "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                    "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                    "실제 조회로 검증됨."
})

print(f"선례 총 {len(PRECEDENT_LIBRARY)}건 (검증 완료된 것만)")
for p in PRECEDENT_LIBRARY:
    print(f"  [{p['rule']}] {p['type']}")

선례 총 7건 (검증 완료된 것만)
  [Thiocarbonyl_group] 긍정_승인약물쌍
  [catechol] 정량_활성데이터
  [hydroxamic_acid] 부정_참고사례_검증필요
  [beta-keto/anhydride] 긍정_통계검증결과
  [Michael_acceptor_1] 위험=메커니즘_참고
  [alkyl_halide] 위험=메커니즘_참고
  [azo_A(324)] 위험=메커니즘_참고_검증완료


In [15]:
# 승인약물(max_phase=4) + 임상실패/철수(withdrawn) 데이터를 함께 확보
def fetch_chembl_by_phase_and_status(max_phase_value=None, withdrawn=None, limit=200):
    filters = {}
    if max_phase_value is not None:
        filters['max_phase'] = max_phase_value
    results = molecule.filter(**filters).only(
        ['molecule_structures', 'pref_name', 'max_phase', 'withdrawn_flag'])[:limit]
    data_list = []
    for r in results:
        struct = r.get('molecule_structures')
        if struct and struct.get('canonical_smiles'):
            data_list.append({
                "smiles": struct['canonical_smiles'],
                "name": r.get('pref_name'),
                "max_phase": r.get('max_phase'),
                "withdrawn": r.get('withdrawn_flag'),
            })
    return data_list

print("승인약물(전체) 가져오는 중...")
approved_drugs_full = fetch_chembl_by_phase_and_status(max_phase_value=4, limit=500)
print(f"  {len(approved_drugs_full)}개 확보")

withdrawn_count = sum(1 for d in approved_drugs_full if d['withdrawn'])
print(f"  이 중 철수(withdrawn) 표시된 것: {withdrawn_count}개")

승인약물(전체) 가져오는 중...
  498개 확보
  이 중 철수(withdrawn) 표시된 것: 76개


In [16]:
withdrawn_drugs = fetch_chembl_by_phase_and_status(max_phase_value=4, limit=500)
withdrawn_only = [d for d in withdrawn_drugs if d['withdrawn']]

withdrawn_matched = []
for d in withdrawn_only:
    mol = Chem.MolFromSmiles(d['smiles'])
    if mol is None:
        continue
    problems = detect_toxicophores(d['smiles'])
    known = [p for p in problems if get_replacement_candidates(p['rule_name']) is not None]
    if known:
        withdrawn_matched.append({"drug": d, "problems": known})

print(f"철수 약물 중 우리 규칙과 매칭되는 것: {len(withdrawn_matched)}개")
for w in withdrawn_matched:
    print(f"  {w['drug']['name']}: {[p['rule_name'] for p in w['problems']]}")

철수 약물 중 우리 규칙과 매칭되는 것: 23개
  TROGLITAZONE: ['thioester']
  TOLRESTAT: ['Thiocarbonyl_group']
  SULFATHIAZOLE: ['aniline']
  SULFAMERAZINE: ['aniline']
  CYCLOBARBITAL: ['beta-keto/anhydride']
  PHENOBARBITAL: ['beta-keto/anhydride']
  SULFAMETHAZINE: ['aniline']
  SECOBARBITAL: ['beta-keto/anhydride']
  PENTOBARBITAL: ['beta-keto/anhydride']
  AMOBARBITAL: ['beta-keto/anhydride']
  HEXOBARBITAL: ['beta-keto/anhydride']
  APROBARBITAL: ['beta-keto/anhydride']
  SULFAMETHOXYPYRIDAZINE: ['aniline']
  ETRETINATE: ['Michael_acceptor_1']
  THALIDOMIDE: ['phthalimide']
  FLUNITRAZEPAM: ['nitro_group']
  PHENYLBUTAZONE: ['beta-keto/anhydride']
  LINDANE: ['alkyl_halide']
  TRICHLOROETHANE: ['alkyl_halide']
  ROSIGLITAZONE: ['thioester']
  ROFECOXIB: ['stilbene']
  PROBUCOL: ['het-C-het_not_in_ring']
  CHLORAMPHENICOL: ['alkyl_halide', 'nitro_group']


In [18]:
from src.tools.molecule_editor import propose_fix, canonicalize, iterative_fix_loop
print("준비 완료")

준비 완료


In [19]:
for w in withdrawn_matched:
    rule = w['problems'][0]['rule_name']
    result = iterative_fix_loop(w['drug']['smiles'], max_iterations=10)
    print(f"\n{w['drug']['name']} (철수 사유 규칙: {rule})")
    print(f"  원본: {w['drug']['smiles']}")
    print(f"  우리 시스템 최종 결과: {result['status']} -> {result['final_smiles']}")


TROGLITAZONE (철수 사유 규칙: thioester)
  원본: Cc1c(C)c2c(c(C)c1O)CCC(C)(COc1ccc(CC3SC(=O)NC3=O)cc1)O2
  우리 시스템 최종 결과: success -> Cc1c(C)c2c(c(C)c1O)CCC(C)(COc1ccc(CC3OC(=O)NC3=O)cc1)O2

TOLRESTAT (철수 사유 규칙: Thiocarbonyl_group)
  원본: COc1ccc2c(C(=S)N(C)CC(=O)O)cccc2c1C(F)(F)F
  우리 시스템 최종 결과: success -> COc1ccc2c(C(=O)N(C)CC(=O)O)cccc2c1C(F)(F)F

SULFATHIAZOLE (철수 사유 규칙: aniline)
  원본: Nc1ccc(S(=O)(=O)Nc2nccs2)cc1
  우리 시스템 최종 결과: success -> CC(=O)Nc1ccc(S(=O)(=O)Nc2nccs2)cc1

SULFAMERAZINE (철수 사유 규칙: aniline)
  원본: Cc1ccnc(NS(=O)(=O)c2ccc(N)cc2)n1
  우리 시스템 최종 결과: success -> CC(=O)Nc1ccc(S(=O)(=O)Nc2nccc(C)n2)cc1

CYCLOBARBITAL (철수 사유 규칙: beta-keto/anhydride)
  원본: CCC1(C2=CCCCC2)C(=O)NC(=O)NC1=O
  우리 시스템 최종 결과: stuck -> CCC1(C2=CCCCC2)C(=O)NC(=O)NC1=O

PHENOBARBITAL (철수 사유 규칙: beta-keto/anhydride)
  원본: CCC1(c2ccccc2)C(=O)NC(=O)NC1=O
  우리 시스템 최종 결과: stuck -> CCC1(c2ccccc2)C(=O)NC(=O)NC1=O

SULFAMETHAZINE (철수 사유 규칙: aniline)
  원본: Cc1cc(C)nc(NS(=O)(=O)c2ccc(N)cc2)n1
  우리 시스템 최종 결과: success ->

In [20]:
print(detect_toxicophores("NC1CCC(=O)NC1=O"))

[{'rule_name': 'phthalimide', 'atom_indices': [1, 3, 4, 5, 6, 7, 8]}]


In [21]:
pattern_check_barb = Chem.MolFromSmarts("C(=O)NC(=O)")
mol_phenobarb = Chem.MolFromSmiles("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O")
print("바르비투레이트, N연결 우레이드 매치:", mol_phenobarb.HasSubstructMatch(pattern_check_barb))

pattern_check_probucol = Chem.MolFromSmarts("[CX4](S)(S)")
mol_probucol = Chem.MolFromSmiles("CC(C)(Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1)Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1")
print("Probucol, S,S 매치:", mol_probucol.HasSubstructMatch(pattern_check_probucol))

바르비투레이트, N연결 우레이드 매치: True
Probucol, S,S 매치: True


In [22]:
pattern_extended = Chem.MolFromSmarts("[CX4]([OX2,SX2])([OX2,SX2])")
mol_probucol_test = Chem.MolFromSmiles("CC(C)(Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1)Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1")
print("확장 패턴 매치:", mol_probucol_test.HasSubstructMatch(pattern_extended))

# 회귀 확인 - 기존 오르토에스터/아세탈도 여전히 잡히는지
mol_orig_test = Chem.MolFromSmiles("COC(OC)C(C)c1ccccc1")
print("기존 아세탈 매치:", mol_orig_test.HasSubstructMatch(pattern_extended))

확장 패턴 매치: True
기존 아세탈 매치: True


In [23]:
pattern_cyclic_imide = Chem.MolFromSmarts("C(=O)NC(=O)")
mol_barb = Chem.MolFromSmiles("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O")
mol_thalido_残 = Chem.MolFromSmiles("NC1CCC(=O)NC1=O")

print("바르비투레이트 매치:", mol_barb.HasSubstructMatch(pattern_cyclic_imide))
print("탈리도마이드 잔여물 매치:", mol_thalido_残.HasSubstructMatch(pattern_cyclic_imide))

matches = mol_barb.GetSubstructMatches(pattern_cyclic_imide)
print("매치 위치:", matches)
for i, idx in enumerate(matches[0]):
    print(f"  위치{i} -> idx{idx}: {mol_barb.GetAtomWithIdx(idx).GetSymbol()}")

바르비투레이트 매치: True
탈리도마이드 잔여물 매치: True
매치 위치: ((9, 10, 11, 12, 13), (12, 13, 14, 15, 16))
  위치0 -> idx9: C
  위치1 -> idx10: O
  위치2 -> idx11: N
  위치3 -> idx12: C
  위치4 -> idx13: O


In [24]:
!cat src/tools/replacement_library.py


REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "al

In [31]:
%%writefile src/tools/replacement_library.py

REPLACEMENT_LIBRARY = {
    "nitro_group": {
        "problem_smarts": "[N+](=O)[O-]",
        "candidates": [
            {"smiles": "N", "name": "primary amine",
             "rationale": "[참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 "
                          "항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 "
                          "치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 "
                          "치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 "
                          "환원성 대사 중간체 생성 경로를 제거함"},
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지"},
            {"smiles": "C#N", "name": "nitrile",
             "rationale": "대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소"},
        ],
    },
    "aldehyde": {
        "problem_smarts": "[CX3H1](=O)",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "알데히드의 친전자성(단백질 부가물 형성 우려)을 제거하면서 유사한 형태 유지"},
            {"smiles": "C(O)", "name": "alcohol",
             "rationale": "가장 단순한 환원형 대체, 반응성 크게 감소"},
        ],
    },
    "Michael_acceptor_1": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=CC(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "saturated (C-C single bond)",
             "rationale": "[참고] 에타크린산처럼 시스테인 잔기와의 공유결합 자체가 "
                          "작용 메커니즘인 공유결합 억제제(covalent inhibitor) "
                          "계열에는 본 경고가 그대로 적용되지 않을 수 있음. || "
                          "알파,베타-불포화 카르보닐의 C=C 이중결합을 환원하여 "
                          "단백질 친전자성 부가반응(Michael addition, covalent "
                          "binding) 위험을 제거함"},
        ],
    },
    "acid_halide": {
        "problem_smarts": "C(=O)[F,Cl,Br,I]",
        "candidates": [
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "고반응성 아실할라이드를 안정적인 아마이드로 대체"},
            {"smiles": "C(=O)O", "name": "ester",
             "rationale": "아마이드보다 극성이 낮고 유연한 대체 옵션, 가수분해 속도 조절 가능 (검증 필요)"},
        ],
    },
    "alkyl_halide": {
        "problem_smarts": "[Cl,Br,I]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "[참고] 메클로르에타민, 사이클로포스파미드, 카머스틴, "
                          "클로람부실 등 알킬화 항암제는 DNA 알킬화(반응성) 자체가 "
                          "세포독성 치료 메커니즘이므로, 이 계열에는 본 치환이 "
                          "적절하지 않음. || 이탈기를 제거해 알킬화 반응성을 없앰, "
                          "극성은 유사하게 유지"},
            {"smiles": "F", "name": "fluorine",
             "rationale": "할로겐을 유지하되 C-F 결합은 강해 이탈기로 작용하지 않음, 입체적 크기도 유사"},
        ],
    },
    "aniline": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NH2]c1ccc([#6,#7,#8,#16])cc1",
        "target_idx_in_pattern": 0,
        "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
        "anchor_indices_in_pattern": (0, 5),
        "candidates": [
            {"edit_type": "add_substituent", "param": "C(=O)C",
             "target_idx_in_pattern": 0,
             "name": "acetamide (acylated amine)",
             "rationale": "[참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 "
                          "프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 "
                          "안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 "
                          "특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 "
                          "배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 "
                          "아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단"},
            {"edit_type": "replace_ring", "param": "[*:1]C12CC(C1)(C2)[*:2]",
             "ring_atom_indices_in_pattern": [1, 2, 3, 4, 6, 7],
             "anchor_indices_in_pattern": (0, 5),
             "name": "BCP (bicyclo[1.1.1]pentane)",
             "rationale": "para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic "
                          "탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive "
                          "metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 "
                          "생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 "
                          "(문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, "
                          "실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 "
                          "변화 폭이 크지만, 물성 개선 효과도 더 큼"},
        ],
    },
    "Sulfonic_acid_2": {
        "problem_smarts": "[#6]S(=O)(=O)[OX2H1,OX1-]",
        "candidates": [
            {"smiles": "S(=O)(=O)N", "name": "sulfonamide",
             "rationale": "[참고] 암페타민 설페이트, 사퀴나비르 메실레이트처럼 "
                          "일부 승인약물에서 설폰산/설폰산 유사기는 활성 골격이 "
                          "아니라 염(salt) 형성을 위한 카운터이온으로만 존재함. "
                          "이 경우 본 규칙이 다루는 '독성 유발 골격'과 무관하므로, "
                          "치환 대상 여부를 판단하기 전에 이 산이 활성 골격의 "
                          "일부인지 염 형성용인지 구분이 필요함. || 생리적 pH에서 "
                          "이온화 정도(전하)를 크게 낮춰 세포막 투과성을 개선함. "
                          "설폰산은 대부분 음이온 상태로 존재해 경구 흡수가 저해되는 "
                          "경우가 많으나, 설폰아마이드는 유사한 골격을 유지하면서도 "
                          "중성에 가까워 약물유사성이 개선됨"},
            {"smiles": "C(=O)O", "name": "carboxylic acid",
             "rationale": "설폰산보다 산성도가 약하고 부피가 작은 산성 bioisostere "
                          "(검증 필요)"},
        ],
    },
    "imine_1_oxime": {
        "edit_method": "atom_edit",
        "problem_smarts": "C=N[OX2H1]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "옥심의 C=N 결합을 환원하여, 가수분해 시 원래의 반응성 "
                          "카르보닐(알데히드/케톤)로 되돌아갈 수 있는 대사 불안정 "
                          "경로를 제거함"},
        ],
    },
    "imine_1_general": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX3;!$(C(N)(N)=N)]=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "amine (reduced)",
             "rationale": "일반 이민(C=N-R)을 환원하여 가수분해 시 반응성 카르보닐로 "
                          "되돌아갈 수 있는 대사 불안정 경로를 제거함. 옥심 특유의 "
                          "메커니즘보다는 근거가 다소 약하며, 하위 구조별 개별 검증 필요. "
                          "구아니딘(N-C(=N)-N, 공명구조로 일반 이민과 반응성이 다름)은 "
                          "이 SMARTS에서 명시적으로 제외함"},
        ],
    },
    "catechol": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H;$(Oc1ccccc1O)]",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 도파민, 에피네프린, 이소프로테레놀 등 카테콜아민류 "
                      "약물은 카테콜 구조 자체가 아드레날린/도파민 수용체 결합에 "
                      "필수적인 약효 골격이므로, 이 경우 본 치환은 독성 감소가 "
                      "아니라 약효 상실로 이어짐. 실제 도파민은 도파민 수용체 "
                      "D1(Ki 4.3-5.6 nM), D2(Ki 4.7-7.2 nM), D3(Ki 6.4-7.3 nM)에 "
                      "단자릿수 나노몰 수준의 강력한 작용제 친화도를 가짐(IUPHAR/BPS "
                      "Guide to PHARMACOLOGY 확인). || 인체의 COMT(catechol-O-"
                      "methyltransferase) 효소가 카테콜을 메톡시페놀로 메틸화하여 "
                      "해독하는 생리적 경로와 동일한 원리. 오르토-퀴논으로의 산화 "
                      "경로를 차단하여 세포독성/유전독성 우려를 낮춤 (학생 확인 "
                      "예정: ScienceDirect catechol overview, PMC6643002 등 참고)"},
         ],
    },
    "Thiocarbonyl_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "[#6]=[#16]",
        "target_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carbonyl (O replacing S)",
             "rationale": "[참고] 티오펜탈·티아밀랄(치오바르비투레이트, C=S가 지용성 "
                          "증가로 빠른 마취효과에 기여)과 티오구아닌(퓨린 유사 항대사물, "
                          "황이 작용기전에 필수)처럼 황 원자가 약효/효력에 직접 "
                          "기여하는 경우가 있어, 이 계열에는 본 치환이 부적절할 수 "
                          "있음. || 황을 산소로 대체(티오카르보닐->카르보닐)하는 것은 "
                          "흔한 bioisostere 전략으로, 갑상선 기능 저해 등 황 함유 "
                          "작용기 특유의 대사/독성 우려를 낮춤 (검증 필요, "
                          "thiourea->urea 치환 논리와 동일 계열)"},
        ],
    },
    "thiol_2": {
        "problem_smarts": "[SX2H1]",
        "candidates": [
            {"smiles": "O", "name": "hydroxyl (alcohol)",
             "rationale": "티올의 금속 킬레이팅 및 산화(이황화물/술펜산 형성) 반응성을 "
                          "제거하면서, 극성·수소결합 특성을 유사하게 유지함"},
            {"smiles": "C(=O)N", "name": "amide",
             "rationale": "티올을 아마이드로 대체하여 반응성을 낮추면서 약물유사 골격에서 "
                          "흔히 쓰이는 안정적 작용기로 전환 (검증 필요)"},
        ],
    },
    "thiol_1_dithiocarbamate": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=S)[SX1-]",
        "candidates": [
            {"edit_type": "replace_multi",
             "param": [
                 {"idx_in_pattern": 1, "new_element": 8, "new_charge": 0},
                 {"idx_in_pattern": 2, "new_element": 7, "new_charge": 0},
             ],
             "name": "carbamate (O,N replacing S,S)",
             "rationale": "디티오카바메이트(R-O-C(=S)-S-)를 카바메이트(R-O-C(=O)-N)로 "
                          "전환. 두 황 원자를 각각 산소·질소로 교체하여 금속 킬레이팅 "
                          "능력과 효소 억제 활성(디티오카바메이트류 특유의 살충제성 "
                          "독성 기전)을 제거함 (검증 필요)"},
        ],
    },
    "thiol_1_thiocarboxylate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX1-]C(=O)",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "carboxylate (O replacing S)",
             "rationale": "티오카르복실산 음이온(R-C(=O)-S-)의 황을 산소로 대체하여 "
                          "카르복실산염(R-C(=O)-O-)으로 전환. 황 원자의 금속 킬레이팅 "
                          "및 친핵성 반응성을 제거함 (검증 필요)"},
        ],
    },
    "het-C-het_not_in_ring": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]([OX2,SX2])([OX2,SX2])",
        "candidates": [
            {"edit_type": "remove_substituent",
             "center_idx_in_pattern": 0,
             "remove_idx_in_pattern": 1,
             "upgrade_bond_to_idx_in_pattern": 2,
             "name": "ketone/ester (one heteroatom substituent removed, C=O formed)",
             "rationale": "아세탈/케탈/오르토에스터(산소 2개) 또는 디티오아세탈(황 2개, "
                          "실제 철수약물 Probucol에서 확인) 등 탄소 하나에 헤테로원자 2개가 "
                          "붙은 구조는 가수분해/해리에 민감하여 반응성 카르보닐로 쉽게 "
                          "전환되며 대사 불안정성을 일으킴. 헤테로원자 하나를 제거하고 "
                          "남은 것을 카르보닐로 승격시켜, 가수분해로 어차피 도달할 안정한 "
                          "최종 형태로 미리 전환함 (검증 필요)"},
        ],
    },
    "cyclic_imide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[C;R](=O)[N;R][C;R](=O)",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (2, 3),
             "name": "ring-opened amide (imide bond cleaved)",
             "rationale": "고리형 이미드(우레이드) 구조는 바르비투레이트류(페노바르비탈, "
                      "펜토바르비탈 등 다수 철수약물에서 실제 확인됨)와 탈리도마이드의 "
                      "잔여 글루타르이미드 고리에서 나타나며, 가수분해에 민감한 반응성 "
                      "구조임. 고리 내 아마이드 결합 하나를 끊어 개환함으로써 실제 "
                      "가수분해의 첫 단계를 근사함. 고리 구성원(R)만 매치하도록 제한하여, "
                      "개환 후 남은 사슬에 재적용되어 조각화되는 것을 방지함 "
                      "(ChEMBL 조회로 검증된 실제 철수약물 다수에서 발견, 검증 필요)"},
        ],
    },
    "hydroquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2H]c1ccc([OX2H,NX3H1,NX3H2])cc1",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "add_substituent", "param": "C", "name": "methoxy",
             "rationale": "[참고] 아세트아미노펜은 정상 용량에서는 안전하며 과다복용 "
                          "시에만 위험한 용량 의존적 사례임. 본 시스템은 치료지수를 "
                          "고려하지 않으므로, 아트로핀·디곡신·와파린처럼 좁은 치료지수를 "
                          "가진 기존 약물 전반에 유사하게 적용되는 한계임. || 파라 "
                          "위치에 OH와 (OH 또는 NH)가 있는 구조(하이드로퀴논/파라-"
                          "아미노페놀 계열)는 산화되어 파라-퀴논 또는 파라-퀴논이민(예: "
                          "아세트아미노펜의 NAPQI)을 형성, 글루타치온 고갈과 단백질 "
                          "공유결합을 통한 간독성 위험이 있음"},
        ],
    },
    "azo_A(324)": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=N",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "hydrazine (reduced)",
             "rationale": "아조기(N=N)는 체내에서 아조환원효소에 의해 환원되어 두 개의 "
                          "방향족 아민으로 분해되며, 그 중 일부(벤지딘류 등)가 발암성을 "
                          "가지는 것으로 잘 알려짐(아조 색소의 대표적 독성 메커니즘). "
                          "이중결합을 환원하여 하이드라진 형태로 전환, 완전한 아민 "
                          "분해 경로 자체를 차단함 (검증 필요: 하이드라진 자체의 "
                          "잔여 반응성은 추가 확인 필요)"},
        ],
    },
    "Three-membered_heterocycle": {
        "edit_method": "atom_edit",
        "problem_smarts": "[CX4]1[OX2][CX4]1",
        "candidates": [
            {"edit_type": "open_epoxide", "break_pair_in_pattern": (1, 2),
             "name": "vicinal diol (ring-opened)",
             "rationale": "에폭시드(3원자 고리, 옥시란)는 고리 변형(strain)으로 인해 "
                          "친핵체(DNA, 단백질)와 쉽게 반응하는 알킬화제로 작용함. "
                          "체내 에폭시드 가수분해효소(epoxide hydrolase)가 실제로 "
                          "수행하는 반응과 동일하게 고리를 열어 비시날 디올(vicinal "
                          "diol)로 전환, 반응성을 제거함"},
        ],
    },
    "diketo_group": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)C(=O)",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alpha-hydroxy ketone (reduced)",
             "rationale": "비시날 알파-디케톤(1,2-diketone)은 반응성이 높은 친전자체로 "
                          "단백질과 부가물을 형성할 수 있으며, 흡입 시 호흡기 독성을 "
                          "일으키는 것으로 알려진 디아세틸(버터향 첨가제) 사례가 대표적임. "
                          "카르보닐 하나를 환원하여 알파-하이드록시케톤(아실로인)으로 "
                          "전환, 케토-환원효소에 의한 실제 해독 경로와 유사한 방향으로 "
                          "반응성을 낮춤 (검증 필요)"},
        ],
    },
    "thioester": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2](C(=O))",
        "target_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "replace_element", "param": 8, "name": "ester (O replacing S)",
             "rationale": "티오에스터의 황을 산소로 대체하여 일반 에스터로 전환. "
                          "티오에스터는 일반 에스터보다 가수분해 반응성이 높고 아실화 "
                          "능력이 강해 단백질 등과 부반응 우려가 있음 (검증 필요)"},
        ],
    },
    "N-nitroso": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2;+0;!$(N(=O)[O-])]=[OX1;+0]",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "N-hydroxylamine (reduced)",
             "rationale": "N-니트로소 화합물(니트로사민)은 대사 활성화(알파-수산화)를 "
                          "거쳐 강력한 알킬화 발암물질을 생성하는 것으로 잘 알려짐 "
                          "(발사르탄, 라니티딘 등 실제 의약품 불순물 리콜 사례). "
                          "N=O를 환원하여 반응성을 낮춤 (검증 필요: 완전한 해독은 "
                          "탈니트로소화가 필요하며 이는 근사적 접근)"},
        ],
    },
    "hydrazine": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX3H2][NX3H1]",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 0,
             "center_idx_in_pattern": 1,
             "name": "amide/amine (terminal N removed)",
             "rationale": "하이드라진/하이드라지드(R-NH-NH2)의 말단 질소를 제거하여 "
                          "단순 아민 또는 아마이드로 되돌림. 하이드라진류는 대사 시 "
                          "반응성 디아제늄 중간체를 형성해 유전독성을 일으킬 수 있는 "
                          "것으로 알려짐. 이는 azo_A(324) 환원 시 생성되는 하이드라진 "
                          "중간체의 잔여 위험을 추가로 낮추는 후속 규칙이기도 함 "
                          "(검증 필요)"},
        ],
    },
    "sulphate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[OX2][SX4](=O)(=O)[OX1,OX2H]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "alcohol (sulfate group removed)",
             "rationale": "알킬 설페이트 에스터(R-O-SO3-)는 대사되어 반응성 있는 "
                          "설페이트 이탈기를 통한 알킬화제로 작용할 수 있음(디메틸설페이트가 "
                          "강력한 발암/독성 물질로 잘 알려진 대표 사례). 설페이트기 전체를 "
                          "제거하여 원래의 알코올로 되돌림 (검증 필요)"},
        ],
    },
    "N_oxide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[n+][O-]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "pyridine (N-oxide removed)",
             "rationale": "방향족 N-옥사이드는 산화적 대사산물이자 반응성 중간체 "
                          "생성 경로의 일부일 수 있음. 산소를 제거하여 원래의 중성 "
                          "방향족 아민(피리딘 등)으로 환원, 자연 대사에서의 환원 "
                          "경로와 유사한 방향으로 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "2-halo_pyridine": {
        "edit_method": "atom_edit",
        "problem_smarts": "n:c(-[Cl,Br,I])",
        "center_idx_in_pattern": 1,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 2,
             "center_idx_in_pattern": 1,
             "name": "pyridine (halogen removed)",
             "rationale": "피리딘 고리 질소에 인접한 위치의 할로겐(특히 불소/염소)은 "
                          "친핵성 방향족 치환(SNAr) 반응에 취약해, 체내 친핵체(글루타치온, "
                          "단백질 시스테인 등)와 반응할 수 있음. 할로겐을 제거하고 수소로 "
                          "대체하여 이 반응성 경로를 차단함 (검증 필요)"},
        ],
    },
    "disulphide": {
        "edit_method": "atom_edit",
        "problem_smarts": "[SX2][SX2]",
        "candidates": [
            {"edit_type": "cleave_bond", "cleave_pair_in_pattern": (0, 1),
             "name": "two thiols (bond cleaved)",
             "rationale": "[참고] 이황화결합(S-S)은 시스틴/단백질의 3차구조 형성에 "
                          "필수적인 정상 생체 구조이기도 하므로, 이 결합이 약물의 "
                          "구조 안정성이나 표적 결합에 관여하는 경우 본 치환이 "
                          "부적절할 수 있음. || 디티오카바메이트류(티우람 등) 농약/"
                          "살균제에서 흔한 반응성 이황화결합을 두 개의 티올로 분리, "
                          "산화·금속킬레이팅 반응성을 낮춤 (검증 필요)"},
        ],
    },
    "quinone_A(370)": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1C=CC(=O)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "hydroquinone (reduced, re-aromatized)",
             "rationale": "파라벤조퀴논은 산화환원 사이클(redox cycling)을 통해 활성산소종(ROS)을 "
                          "생성하고 DNA/단백질과 직접 공유결합하는 대표적 반응성 구조. 체내 "
                          "NQO1(퀴논 환원효소) 효소가 실제로 수행하는 반응과 동일하게 두 카르보닐을 "
                          "환원하고 고리를 재방향족화하여 안정적인 하이드로퀴논으로 전환. 결과물이 "
                          "다시 hydroquinone 규칙에 해당할 수 있으며, 이 경우 반복 루프가 자동으로 "
                          "메톡시페놀 등 산화에 더 안정적인 형태로 한 단계 더 개선함 (검증 필요, "
                          "안트라퀴논 등 융합고리형은 미지원)"},
        ],
    },
    "quinone_A_anthraquinone": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1c2ccccc2C(=O)c2ccccc21",
        "target_pairs_in_pattern": [(1, 0), (8, 9)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 5, 6, 7, 8],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "anthrahydroquinone (reduced, re-aromatized)",
             "rationale": "안트라퀴논은 벤조퀴논과 동일한 산화환원 사이클링(redox cycling) 메커니즘을 "
                          "가지되, 두 벤젠 고리에 의해 안정화되어 항암제(독소루비신 등) 및 염료에서도 "
                          "흔히 쓰이는 골격임. 두 카르보닐을 동시에 환원하고 중앙 고리를 재방향족화하여 "
                          "안트라하이드로퀴논으로 전환, 산화환원 사이클링 능력을 제거함. 결과물이 "
                          "hydroquinone 규칙에 해당할 수 있어 반복 루프가 자동으로 추가 개선 가능 "
                          "(Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "quinone_diimine": {
        "edit_method": "atom_edit",
        "problem_smarts": "N=C1C=CC(=N)C=C1",
        "target_pairs_in_pattern": [(1, 0), (4, 5)],
        "ring_atoms_in_pattern": [1, 2, 3, 4, 6, 7],
        "ring_bonds_in_pattern": [(1, 2), (2, 3), (3, 4), (4, 6), (6, 7), (7, 1)],
        "candidates": [
            {"edit_type": "reduce_multi_bond", "name": "phenylenediamine (reduced, re-aromatized)",
             "rationale": "퀴논디이민(quinone diimine)은 벤조퀴논의 산소가 이민으로 치환된 유사체로, "
                          "동일한 산화환원 사이클링 메커니즘을 가지며 헤어염료 성분(파라페닐렌디아민 "
                          "산화형) 등에서 피부 알레르기 및 접촉성 피부염을 유발하는 것으로 알려짐. 두 "
                          "이민을 동시에 환원하고 고리를 재방향족화하여 페닐렌디아민(원래의 안정한 "
                          "환원형)으로 전환 (Murcko scaffold 분석으로 발견, 검증 필요)"},
        ],
    },
    "isocyanate": {
        "edit_method": "atom_edit",
        "problem_smarts": "[NX2]=[CX2]=[OX1]",
        "center_idx_in_pattern": 0,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 0,
             "name": "amine (NCO hydrolyzed)",
             "rationale": "이소시아네이트(R-N=C=O)는 매우 반응성이 높은 친전자체로, "
                          "단백질/아미노기와 쉽게 부가반응을 일으켜 직업성 천식·과민증을 "
                          "유발하는 것으로 잘 알려짐(TDI, MDI 등 산업용 이소시아네이트 "
                          "사례). 체내/환경에서 실제로 일어나는 가수분해 경로(R-NCO + H2O "
                          "-> R-NH2 + CO2)와 동일하게 카르보닐 탄소와 산소를 제거하고 "
                          "질소만 남겨 아민으로 전환 (검증 필요)"},
        ],
    },
    "triple_bond": {
        "problem_smarts": "C#C",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (0, 1),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "alkene (partially reduced)",
             "rationale": "말단 알카인(삼중결합)은 CYP450 효소에 의해 기계기반 억제"
                          "(mechanism-based inhibition) 경로로 대사되며, 반응성 케텐/"
                          "에폭사이드 중간체를 형성해 효소를 비가역적으로 불활성화할 "
                          "수 있음(에티닐에스트라디올 등에서 알려진 메커니즘). 삼중결합을 "
                          "이중결합으로 환원하여 반응성을 낮춤 (검증 필요, 완전 포화가 "
                          "아닌 부분 환원)"},
        ],
    },
    "stilbene": {
        "problem_smarts": "c-[CX3]=[CX3]-c",
        "edit_method": "atom_edit",
        "target_idx_pair_in_pattern": (1, 2),
        "candidates": [
            {"edit_type": "reduce_bond", "name": "diarylethane (reduced)",
             "rationale": "스틸벤 구조(두 방향족 고리를 잇는 C=C)는 디에틸스틸베스트롤"
                          "(DES)처럼 내분비교란 및 대사 산화를 통한 반응성 중간체 형성이 "
                          "알려진 골격. 이중결합을 환원하여 평면성을 낮추고 대사 반응성을 "
                          "완화함 (검증 필요, 에스트로겐 수용체 결합에 필요한 형태 자체를 "
                          "훼손할 수 있어 신중한 해석 필요)"},
        ],
    },
    "beta-keto/anhydride": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)OC(=O)",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 3,
             "center_idx_in_pattern": 2,
             "name": "carboxylic acid (anhydride hydrolyzed)",
             "rationale": "산 무수물(R-C(=O)-O-C(=O)-R')은 강한 아실화제로 단백질 아미노산 "
                          "잔기와 쉽게 반응하며, 수용액 환경에서 자발적으로 가수분해되어 "
                          "두 개의 카르복실산으로 분해되는 것이 자연스러운 무독화 경로임. "
                          "한쪽 아실기를 제거하여 이 가수분해 최종형(카르복실산)으로 직접 "
                          "전환 (검증 필요). ※ 대안 후보(무수물->아마이드/이미드 bioisostere) "
                          "는 문헌 확인 후 추가 예정"},
        ],
    },
    "phthalimide": {
        "edit_method": "atom_edit",
        "problem_smarts": "O=C1c2ccccc2C(=O)N1[#6]",
        "center_idx_in_pattern": 10,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 1,
             "center_idx_in_pattern": 10,
             "name": "primary amine (imide hydrolyzed)",
             "rationale": "프탈이미드(고리형 이미드)는 탈리도마이드 등에서 알려진 골격으로, "
                          "체내에서 가수분해되어 원래의 1차 아민과 프탈산으로 분해되는 것이 "
                          "자연스러운 대사 경로임. 이 가수분해 용이성 자체가 대사 불안정성/"
                          "반응성 우려의 근거이며, 고리 전체를 제거하여 이 가수분해 최종형인 "
                          "1차 아민으로 직접 전환 (검증 필요)"},
        ],
    },
    "hydroxamic_acid": {
        "edit_method": "atom_edit",
        "problem_smarts": "C(=O)N[OX2H1]",
        "center_idx_in_pattern": 2,
        "candidates": [
            {"edit_type": "remove_atom",
             "remove_idx_in_pattern": 3,
             "center_idx_in_pattern": 2,
             "name": "amide (N-hydroxyl removed)",
             "rationale": "[참고] 하이드록삼산(R-C(=O)-NH-OH)은 보리노스타트, 파노비노스타트 "
                          "등 HDAC 억제제에서 아연 킬레이션을 통한 핵심 약효 작용기로 쓰이므로, "
                          "이 계열에는 본 치환이 약효 상실로 이어질 수 있음. || 하이드록삼산은 "
                          "로센 재배열(Lossen rearrangement)을 통해 반응성 이소시아네이트로 "
                          "전환될 수 있는 잠재적 위험이 있음. N-하이드록실기를 제거해 단순 "
                          "아마이드로 전환, 이 재배열 경로를 차단함 (검증 필요)"},
        ],
    },
}

def get_replacement_candidates(rule_name: str) -> dict | None:
    """rule_name에 해당하는 치환 정보(SMARTS + 후보 리스트)를 반환. 없으면 None."""
    return REPLACEMENT_LIBRARY.get(rule_name)


Overwriting src/tools/replacement_library.py


In [26]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix, iterative_fix_loop

print("Probucol:", propose_fix("CC(C)(Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1)Sc1cc(C(C)(C)C)c(O)c(C(C)(C)C)c1", "het-C-het_not_in_ring", candidate_idx=0))
print("\n페노바르비탈:", iterative_fix_loop("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O", max_iterations=10)['status'])
print("탈리도마이드 잔여물:", propose_fix("NC1CCC(=O)NC1=O", "cyclic_imide", candidate_idx=0))

print("\n=== 회귀 테스트 ===")
print(propose_fix("COC(OC)C(C)c1ccccc1", "het-C-het_not_in_ring", candidate_idx=0))

Probucol: {'new_smiles': 'CC(C)=S', 'candidate_used': 'ketone/ester (one heteroatom substituent removed, C=O formed)', 'rationale': '아세탈/케탈/오르토에스터(산소 2개) 또는 디티오아세탈(황 2개, 실제 철수약물 Probucol에서 확인) 등 탄소 하나에 헤테로원자 2개가 붙은 구조는 가수분해/해리에 민감하여 반응성 카르보닐로 쉽게 전환되며 대사 불안정성을 일으킴. 헤테로원자 하나를 제거하고 남은 것을 카르보닐로 승격시켜, 가수분해로 어차피 도달할 안정한 최종 형태로 미리 전환함 (검증 필요)', 'is_valid': True}

페노바르비탈: stuck
탈리도마이드 잔여물: {'new_smiles': 'NC(=O)CCC(N)C=O', 'candidate_used': 'ring-opened amide (imide bond cleaved)', 'rationale': '고리형 이미드(우레이드) 구조는 바르비투레이트류(페노바르비탈, 펜토바르비탈 등 다수 철수약물에서 실제 확인됨)와 탈리도마이드의 잔여 글루타르이미드 고리에서 나타나며, 가수분해에 민감한 반응성 구조임. 고리 내 아마이드 결합 하나를 끊어 개환함으로써 실제 가수분해의 첫 단계를 근사함 (ChEMBL 조회로 검증된 실제 철수약물 다수에서 발견, 검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'CC(C=O)c1ccccc1', 'candidate_used': 'ketone/ester (one heteroatom substituent removed, C=O formed)', 'rationale': '아세탈/케탈/오르토에스터(산소 2개) 또는 디티오아세탈(황 2개, 실제 철수약물 Probucol에서 확인) 등 탄소 하나에 헤테로원자 2개가 붙은 구조는 가수분해/해리에 민감하여 반응성 카르보닐로 쉽게 전환되며 대사 불안정성을 일으킴. 헤테로원자 하나를 

In [27]:
problems_phenobarb = detect_toxicophores("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O")
print(problems_phenobarb)

known_phenobarb = [p for p in problems_phenobarb if get_replacement_candidates(p['rule_name']) is not None]
print("known:", [p['rule_name'] for p in known_phenobarb])

# cyclic_imide 직접 시도
print(propose_fix("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O", "cyclic_imide", candidate_idx=0))

[{'rule_name': 'beta-keto/anhydride', 'atom_indices': [2, 9, 10, 15, 16]}]
known: ['beta-keto/anhydride']
{'new_smiles': 'CCC(C(N)=O)(C(=O)NC=O)c1ccccc1', 'candidate_used': 'ring-opened amide (imide bond cleaved)', 'rationale': '고리형 이미드(우레이드) 구조는 바르비투레이트류(페노바르비탈, 펜토바르비탈 등 다수 철수약물에서 실제 확인됨)와 탈리도마이드의 잔여 글루타르이미드 고리에서 나타나며, 가수분해에 민감한 반응성 구조임. 고리 내 아마이드 결합 하나를 끊어 개환함으로써 실제 가수분해의 첫 단계를 근사함 (ChEMBL 조회로 검증된 실제 철수약물 다수에서 발견, 검증 필요)', 'is_valid': True}


In [28]:
!cat src/tools/toxicophore_detector.py

from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return

In [32]:
%%writefile src/tools/toxicophore_detector.py
from rdkit import Chem
from rdkit.Chem import FilterCatalog

def _build_catalog():
    params = FilterCatalog.FilterCatalogParams()
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    return FilterCatalog.FilterCatalog(params)

_catalog = _build_catalog()
_oxime_pattern = Chem.MolFromSmarts("C=N[OX2H1]")
_guanidine_pattern = Chem.MolFromSmarts("[$(C(N)(N)=N)]")
_cyclic_imide_pattern = Chem.MolFromSmarts("[C;R](=O)[N;R][C;R](=O)")

_DUPLICATE_RULE_MAP = {
    "catechol_A(92)": "catechol",
    "diazo_group": "azo_A(324)",
    "oxime_1": "imine_1_oxime",
    "chinone_1": "quinone_A(370)",
}


def _refine_imine1(mol, atom_indices):
    """imine_1은 옥심(C=N-OH), 구아니딘(N-C(=N)-N), 일반 이민(C=N-R)을
    모두 포함하는 넓은 카테고리이므로, 실제 매치 부분의 화학적 맥락을
    확인해 이름을 세분화한다."""
    if mol.HasSubstructMatch(_oxime_pattern):
        matches = mol.GetSubstructMatches(_oxime_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_oxime"
    if mol.HasSubstructMatch(_guanidine_pattern):
        matches = mol.GetSubstructMatches(_guanidine_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "imine_1_guanidine"
    return "imine_1_general"


def _refine_thiol1(mol, atom_indices):
    """thiol_1은 FilterCatalog 원본이 음이온 황 원자 1개만 매치하는 넓은
    규칙이므로, 그 황이 붙은 탄소의 나머지 결합을 확인해 세분화한다:
    이웃 탄소가 C=S도 가지면 디티오카바메이트, C=O를 가지면
    티오카르복실산염, 둘 다 아니면 일반형(미지원)으로 분류한다."""
    s_idx = atom_indices[0]
    s_atom = mol.GetAtomWithIdx(s_idx)
    for nbr in s_atom.GetNeighbors():
        for nbr2 in nbr.GetNeighbors():
            if nbr2.GetIdx() == s_idx:
                continue
            bond2 = mol.GetBondBetweenAtoms(nbr.GetIdx(), nbr2.GetIdx())
            if bond2 is None or bond2.GetBondTypeAsDouble() != 2.0:
                continue
            if nbr2.GetSymbol() == 'S':
                return "thiol_1_dithiocarbamate"
            if nbr2.GetSymbol() == 'O':
                return "thiol_1_thiocarboxylate"
    return "thiol_1_general"


def _refine_beta_keto_anhydride(mol, atom_indices):
    """FilterCatalog의 beta-keto/anhydride는 산소로 연결된 진짜 무수물과
    질소로 연결된 고리형 이미드(우레이드, 바르비투레이트류 등)를 모두
    포함하는 넓은 카테고리이므로, 두 카르보닐 사이 연결원자를 확인해
    세분화한다."""
    if mol.HasSubstructMatch(_cyclic_imide_pattern):
        matches = mol.GetSubstructMatches(_cyclic_imide_pattern)
        for match in matches:
            if set(match) & set(atom_indices):
                return "cyclic_imide"
    return "beta-keto/anhydride"


def detect_toxicophores(smiles: str) -> list[dict]:
    """
    분자의 SMILES를 받아, FilterCatalog(PAINS+BRENK)에 매치되는
    문제 구조(toxicophore)들을 찾아서 규칙 이름과 해당 원자 인덱스를 반환.
    imine_1은 옥심/구아니딘/일반이민, thiol_1은 디티오카바메이트/
    티오카르복실산염/일반형, beta-keto/anhydride는 산소연결(진짜 무수물)/
    질소연결(고리형 이미드) 하위형으로 세분화하여 반환한다.
    aniline은 FilterCatalog의 단순 [NH2] 탐지 대신, replacement_library의
    확장된 패턴(para-치환 벤젠 포함)을 그대로 사용해 재정의한다.
    PAINS/BRENK가 동일하거나 부분적으로 겹치는 구조를 서로 다른 이름/원자
    범위로 중복 보고하는 경우, 같은 rule_name에 원자 인덱스가 하나라도
    겹치면 중복으로 간주해 제거한다(완전히 동일한 인덱스일 필요는 없음).
    imine_1_general과 isocyanate가 같은 원자(누적이중결합)를 가리키면
    처리 가능한 isocyanate를 우선하고 imine_1_general은 제거한다.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []

    results = []

    for entry in _catalog.GetMatches(mol):
        for fm in entry.GetFilterMatches(mol):
            atom_indices = sorted(set(mol_idx for _, mol_idx in fm.atomPairs))
            rule_name = entry.GetDescription()

            if rule_name == "imine_1":
                rule_name = _refine_imine1(mol, atom_indices)
            elif rule_name == "thiol_1":
                rule_name = _refine_thiol1(mol, atom_indices)
            elif rule_name == "beta-keto/anhydride":
                rule_name = _refine_beta_keto_anhydride(mol, atom_indices)
            elif rule_name == "aniline":
                continue
            elif rule_name in _DUPLICATE_RULE_MAP:
                rule_name = _DUPLICATE_RULE_MAP[rule_name]

            is_duplicate = any(
                r['rule_name'] == rule_name and set(r['atom_indices']) & set(atom_indices)
                for r in results
            )
            if is_duplicate:
                continue

            results.append({
                "rule_name": rule_name,
                "atom_indices": atom_indices,
            })

    from src.tools.replacement_library import get_replacement_candidates
    aniline_info = get_replacement_candidates("aniline")
    if aniline_info:
        aniline_pattern = Chem.MolFromSmarts(aniline_info["problem_smarts"])
        if mol.HasSubstructMatch(aniline_pattern):
            matches = mol.GetSubstructMatches(aniline_pattern)
            for match in matches:
                atom_indices = sorted(set(match))
                results.append({
                    "rule_name": "aniline",
                    "atom_indices": atom_indices,
                })

    isocyanate_atom_sets = [set(r['atom_indices']) for r in results if r['rule_name'] == 'isocyanate']
    if isocyanate_atom_sets:
        results = [
            r for r in results
            if not (r['rule_name'] == 'imine_1_general'
                    and any(set(r['atom_indices']) & iso_set for iso_set in isocyanate_atom_sets))
        ]

    return results

Overwriting src/tools/toxicophore_detector.py


In [34]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.toxicophore_detector)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.molecule_editor import iterative_fix_loop, propose_fix

result_phenobarb3 = iterative_fix_loop("CCC1(c2ccccc2)C(=O)NC(=O)NC1=O", max_iterations=10)
print("페노바르비탈 상태:", result_phenobarb3['status'])
for h in result_phenobarb3['history']:
    print(h)

print("\n탈리도마이드 잔여물:", propose_fix("NC1CCC(=O)NC1=O", "cyclic_imide", candidate_idx=0))

print("\n=== 회귀 테스트 ===")
print(propose_fix("CC(=O)OC(C)=O", "beta-keto/anhydride", candidate_idx=0))

페노바르비탈 상태: stuck
{'step': 0, 'smiles': 'CCC1(c2ccccc2)C(=O)NC(=O)NC1=O', 'problems': [{'rule_name': 'cyclic_imide', 'atom_indices': [2, 9, 10, 15, 16]}]}
{'step': 1, 'smiles': 'CCC(C(N)=O)(C(=O)NC=O)c1ccccc1', 'fixed_rule': 'cyclic_imide', 'problem_reason': '규칙 기반(리스트 순서대로)', 'candidate_used': 'ring-opened amide (imide bond cleaved)', 'candidate_reason': '규칙 기반(고정 인덱스)', 'problems': [{'rule_name': 'aldehyde', 'atom_indices': [9, 10]}, {'rule_name': 'beta-keto/anhydride', 'atom_indices': [2, 3, 5, 6, 7]}]}

탈리도마이드 잔여물: {'new_smiles': 'NC(=O)CCC(N)C=O', 'candidate_used': 'ring-opened amide (imide bond cleaved)', 'rationale': '고리형 이미드(우레이드) 구조는 바르비투레이트류(페노바르비탈, 펜토바르비탈 등 다수 철수약물에서 실제 확인됨)와 탈리도마이드의 잔여 글루타르이미드 고리에서 나타나며, 가수분해에 민감한 반응성 구조임. 고리 내 아마이드 결합 하나를 끊어 개환함으로써 실제 가수분해의 첫 단계를 근사함. 고리 구성원(R)만 매치하도록 제한하여, 개환 후 남은 사슬에 재적용되어 조각화되는 것을 방지함 (ChEMBL 조회로 검증된 실제 철수약물 다수에서 발견, 검증 필요)', 'is_valid': True}

=== 회귀 테스트 ===
{'new_smiles': 'CC(=O)O', 'candidate_used': 'carboxylic acid (anhydride hydroly

In [35]:
current_step1 = "CCC(C(N)=O)(C(=O)NC=O)c1ccccc1"

print("aldehyde 시도:", propose_fix(current_step1, "aldehyde", candidate_idx=0))
print("aldehyde 대안:", propose_fix(current_step1, "aldehyde", candidate_idx=1))
print("beta-keto/anhydride 시도:", propose_fix(current_step1, "beta-keto/anhydride", candidate_idx=0))

aldehyde 시도: None
aldehyde 대안: None
beta-keto/anhydride 시도: None


In [36]:
info_ald = get_replacement_candidates("aldehyde")
pattern_ald = Chem.MolFromSmarts(info_ald['problem_smarts'])
mol_step1 = Chem.MolFromSmiles(current_step1)
print("매치:", mol_step1.HasSubstructMatch(pattern_ald))
print("매치 위치:", mol_step1.GetSubstructMatches(pattern_ald))

from src.tools.molecule_editor import find_core_and_target
located = find_core_and_target(current_step1, "aldehyde")
print("find_core_and_target 결과:", located)

매치: True
매치 위치: ((9, 10),)
find_core_and_target 결과: None


In [38]:
from rdkit.Chem import rdMMPA

frags_debug = rdMMPA.FragmentMol(Chem.MolFromSmiles(current_step1), maxCuts=1, resultsAsMols=False)
for core, chain in frags_debug:
    print(f"core={core!r}, chain={chain!r}")

core='', chain='C[*:1].NC(=O)C(C[*:1])(C(=O)NC=O)c1ccccc1'
core='', chain='CC[*:1].NC(=O)C(C(=O)NC=O)(c1ccccc1)[*:1]'
core='', chain='CCC(C(=O)NC=O)(c1ccccc1)[*:1].NC(=O)[*:1]'
core='', chain='CCC(C(N)=O)(c1ccccc1)[*:1].O=CNC(=O)[*:1]'
core='', chain='CCC(C(N)=O)(C(=O)NC=O)[*:1].c1ccc([*:1])cc1'


In [39]:
!git add -A
!git commit -m "Add cyclic_imide rule (ring-opening for barbiturate-type ureide structures, ring-membership constrained SMARTS to prevent re-application fragmentation) and extend het-C-het_not_in_ring to cover dithioacetals (S,S) alongside acetals/orthoesters (O,O). Refine beta-keto/anhydride detection to distinguish true anhydrides (O-linked) from cyclic imides (N-linked, e.g. barbiturates). Validated against 23 ChEMBL-confirmed withdrawn drugs matching our rule library: Probucol and thalidomide's residual glutarimide ring fully resolved; barbiturates resolve their core reactive ureide ring in step 1, with a secondary formamide-type aldehyde byproduct remaining unresolved due to MMPA fragment-cut limitations (documented). Library now 35 rules."
!git push origin main

[main 407ee9d] Add cyclic_imide rule (ring-opening for barbiturate-type ureide structures, ring-membership constrained SMARTS to prevent re-application fragmentation) and extend het-C-het_not_in_ring to cover dithioacetals (S,S) alongside acetals/orthoesters (O,O). Refine beta-keto/anhydride detection to distinguish true anhydrides (O-linked) from cyclic imides (N-linked, e.g. barbiturates). Validated against 23 ChEMBL-confirmed withdrawn drugs matching our rule library: Probucol and thalidomide's residual glutarimide ring fully resolved; barbiturates resolve their core reactive ureide ring in step 1, with a secondary formamide-type aldehyde byproduct remaining unresolved due to MMPA fragment-cut limitations (documented). Library now 35 rules.
 2 files changed, 41 insertions(+), 8 deletions(-)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 1.96 KiB | 1.96 Mi